# 🌐 PATH B1: Self-Host Gemma-4 E4B trên Kaggle + ngrok Tunnel

- **LoRA Adapter:** `hung2903/gemma-4-E4B-vaccine-xai-merged`
- **Base Model:** `unsloth/gemma-4-E4B-it`
- **FastAPI + ngrok server**

> ⚠️ **LƯU Ý TRÊN KAGGLE:**
> 1. Bật **Internet** trong mục Session Options.
> 2. Chọn Accelerator là **GPU T4 x2**.

In [1]:
%%capture
# Cài đặt Unsloth phiên bản chuẩn dành cho Kaggle
!pip install -q "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps xformers "trl<0.9.0" peft accelerate bitsandbytes
!pip install -q fastapi uvicorn pyngrok nest-asyncio pydantic

In [2]:
from kaggle_secrets import UserSecretsClient
import sys

try:
    secrets = UserSecretsClient()
    HF_TOKEN = secrets.get_secret("HF_TOKEN")
    NGROK_TOKEN = secrets.get_secret("NGROK_TOKEN")
    print("✅ Kaggle Secrets loaded thành công!")
except Exception as e:
    print("❌ Lỗi nạp Secrets! Hãy chắc chắn bạn đã Add-ons -> Secrets -> lưu HF_TOKEN và NGROK_TOKEN.")
    print(f"Chi tiết lỗi: {e}")

✅ Kaggle Secrets loaded thành công!


In [3]:
import os
# ⚠️ Tối ưu phân mảnh bộ nhớ của PyTorch để tránh lỗi OOM
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
# Ép Kaggle chỉ nhận 1 GPU duy nhất thay vì 2 để tránh chia sẻ bộ nhớ lỗi.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import torch
import gc
from unsloth import FastModel
from unsloth.chat_templates import get_chat_template

# Dọn dẹp rác trong RAM/VRAM từ các lần chạy lỗi trước đó
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
else:
    raise RuntimeError("❌ KHÔNG TÌM THẤY GPU! Vui lòng kiểm tra lại cấu hình Accelerator là GPU T4.")

LORA_ADAPTER = "hung2903/gemma-4-E4B-vaccine-xai-merged"
MAX_SEQ_LENGTH = 1024 # Giảm xuống 1024 để tiết kiệm VRAM

print("⏳ Loading Gemma-4 E4B + LoRA adapter...")
model, tokenizer = FastModel.from_pretrained(
    model_name=LORA_ADAPTER,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=torch.float16,
    load_in_4bit=True,
    token=HF_TOKEN,
    device_map="auto",
)

FastModel.for_inference(model)

# Sửa lỗi template gemma-4 không tồn tại
try:
    tokenizer = get_chat_template(tokenizer, chat_template="gemma2")
except:
    tokenizer = get_chat_template(tokenizer, chat_template="gemma")

print(f"✅ Loaded. GPU memory: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

_device = next(model.parameters()).device
print(f"Model device: {_device}")

# Warmup với cấu trúc chat template chuẩn
messages = [{"role": "user", "content": "test"}]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

dummy_enc = tokenizer(text=prompt, return_tensors="pt").to(_device)

with torch.no_grad():
    _ = model.generate(
        input_ids=dummy_enc["input_ids"],
        attention_mask=dummy_enc["attention_mask"],
        max_new_tokens=10,
        use_cache=True,
    )
print("✅ Model warmed up")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
⏳ Loading Gemma-4 E4B + LoRA adapter...
==((====))==  Unsloth 2026.5.7: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/16.0G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/2130 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/203 [00:00<?, ?B/s]

processor_config.json:   0%|          | 0.00/1.69k [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/2.38k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.34k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

✅ Loaded. GPU memory: 10.02 GB
Model device: cuda:0
✅ Model warmed up


In [4]:
# ═══════════════════════════════════════════════════════════════
# Parser ANSWER-FIRST v3 — đồng bộ logic với các notebook khác
# ═══════════════════════════════════════════════════════════════
import re as _re

REV_MISINFO_ORDER = [
    ('khong tin gia',1),('không tin giả',1),('khong sai',1),('không sai',1),
    ('dung su that',1),('đúng sự thật',1),('chinh xac',1),('chính xác',1),
    ('accurate',1),('khong lien quan',1),('không liên quan',1),
    ('tin gia',0),('tin giả',0),('tin sai lech',0),('tin sai lệch',0),
    ('misinformation',0),('sai su that',0),('sai sự thật',0),
]
REV_STANCE = {
    'ung ho':0,'ủng hộ':0,'support':0,'tan thanh':0,'tán thành':0,
    'phan doi':1,'phản đối':1,'oppose':1,'chong':1,'chống':1,
    'trung lap':2,'trung lập':2,'neutral':2,
}
REV_SENTIMENT = {
    'tieu cuc':0,'tiêu cực':0,'negative':0,
    'trung tinh':1,'trung tính':1,'trung hin':1,'neutral':1,
    'trung lap':1,'trung lập':1,
    'tich cuc':2,'tích cực':2,'positive':2,
}

LABEL_MISINFO  = {0: 'Tin giả', 1: 'Chính xác'}
LABEL_STANCE   = {0: 'Ủng hộ', 1: 'Phản đối', 2: 'Trung lập'}
LABEL_SENTIMENT= {0: 'Tiêu cực', 1: 'Trung tính', 2: 'Tích cực'}

def build_prompt(text: str) -> list:
    content = (
        f'Phân tích nội dung sau về vaccine:\n\n"{text[:1000]}"\n\n'
        'QUY TẮC PHÂN LOẠI (bắt buộc tuân thủ):\n'
        '- Misinformation CHỈ được chọn 1 trong 2: "Tin gia" hoặc "Chinh xac". '
        'TUYỆT ĐỐI KHÔNG dùng từ khác. Nội dung không liên quan vaccine '
        'hoặc không có thông tin sai = "Chinh xac".\n'
        '- Stance CHỈ: "Ung ho" / "Phan doi" / "Trung lap".\n'
        '- Sentiment CHỈ: "Tieu cuc" / "Trung tinh" / "Tich cuc".\n\n'
        'Trả lời theo ĐÚNG cấu trúc: KẾT QUẢ trước, GIẢI THÍCH sau.\n'
        '=== KẾT QUẢ ===\n'
        '- Misinformation: <Tin gia HOẶC Chinh xac>\n'
        '- Stance: <Ung ho HOẶC Phan doi HOẶC Trung lap>\n'
        '- Sentiment: <Tieu cuc HOẶC Trung tinh HOẶC Tich cuc>\n'
        '=== GIẢI THÍCH ===\n'
        '<lý luận chi tiết bằng tiếng Việt>'
    )
    return [{"role": "user", "content": content}]

def _mis_binary(seg):
    s = (seg or '').strip().lower()
    if not s: return -1, False
    for conj in [' nhưng ', ' tuy nhiên ', ' nhưng,', ' song ']:
        if conj in s:
            s = s.split(conj, 1)[1]
            break
    NEG_WORDS = ['không', 'khong', 'chẳng', 'chả ', 'chưa ', 'no ']
    for kw, val in REV_MISINFO_ORDER:
        idx = s.find(kw)
        if idx == -1: continue
        prefix = s[:idx]
        has_neg = any(neg in prefix for neg in NEG_WORDS)
        if has_neg:
            return (1 - val), True
        return val, True
    return 1, True

def _match_longest(seg, rev):
    s = (seg or '').lower()
    for k in sorted(rev, key=len, reverse=True):
        if k in s:
            return rev[k], True
    return -1, False

def _seg_after_keyword(line, keyword):
    low = line.lower()
    idx = low.find(keyword.rstrip(':'))
    if idx == -1: return None
    after_kw = line[idx + len(keyword.rstrip(':')):]  
    after_kw_clean = _re.sub(r'^[*\s]*\([^)]*\)', '', after_kw)
    if ':' in after_kw_clean:
        decision = after_kw_clean.split(':', 1)[1]
    else:
        decision = after_kw_clean
    for marker in ['->', '→', 'kết luận:', 'ket luan:', 'phân loại:', 'phan loai:']:
        if marker in decision.lower():
            decision = decision.lower().split(marker, 1)[1][:120]
            break
    mt = _re.search(r'\[([^\]]{1,40})\]', decision)
    if mt: return mt.group(1)
    return decision[:200]

def _parse_block(block_text):
    lines = [l.strip() for l in block_text.split('\n')]
    def find(pfx):
        for ln in lines:
            if pfx in ln: return ln.split(pfx, 1)[-1]
        return None
    seg_m  = find('misinformation:')
    seg_st = find('stance:')
    seg_se = find('sentiment:')
    n = sum(1 for x in [seg_m, seg_st, seg_se] if x is not None)
    m,  mo  = _mis_binary(seg_m)          if seg_m  is not None else (-1, False)
    st, so  = _match_longest(seg_st, REV_STANCE)    if seg_st is not None else (-1, False)
    se, seo = _match_longest(seg_se, REV_SENTIMENT) if seg_se is not None else (-1, False)
    return m, st, se, (mo and so and seo), n

def parse_output(text):
    t = (text or "")
    t_low = t.lower()
    markers = list(_re.finditer(
        r'(?:=== kết quả ===|=== ket qua ===|^kết quả:|^ket qua:)',
        t_low, _re.M))
    if markers:
        blocks = []
        for i, mt in enumerate(markers):
            start = mt.end()
            end = markers[i+1].start() if i+1 < len(markers) else len(t_low)
            seg = t_low[start:end]
            for gm in ['=== giải thích ===','=== giai thich ===','giải thích:','giai thich:']:
                if gm in seg:
                    seg = seg.split(gm, 1)[0]; break
            blocks.append(seg)
        best = None
        for blk in blocks:
            res = _parse_block(blk)
            if best is None or (res[3] and not best[3]) or \
               (res[3]==best[3] and res[4] > best[4]):
                best = res
        if best and best[3]:
            return best[0], best[1], best[2], True
    
    lines = t.split('\n')
    seg_m = seg_st = seg_se = None
    for ln in lines:
        if 'misinformation' in ln.lower() and seg_m  is None and len(ln) < 300:
            seg_m  = _seg_after_keyword(ln, 'misinformation:')
        if 'stance'         in ln.lower() and seg_st is None and len(ln) < 300:
            seg_st = _seg_after_keyword(ln, 'stance:')
        if 'sentiment'      in ln.lower() and seg_se is None and len(ln) < 300:
            seg_se = _seg_after_keyword(ln, 'sentiment:')
    m,  mo  = _mis_binary(seg_m)          if seg_m  is not None else (-1, False)
    st, so  = _match_longest(seg_st, REV_STANCE)    if seg_st is not None else (-1, False)
    se, seo = _match_longest(seg_se, REV_SENTIMENT) if seg_se is not None else (-1, False)
    if mo and so and seo:
        return m, st, se, True
    if (mo + so + seo) >= 2:
        return m, st, se, True
    return m, st, se, False

assert _mis_binary(' tin gia')[0] == 0
assert _mis_binary(' chinh xac')[0] == 1
assert _mis_binary(' khong tin gia')[0] == 1
print("✅ Parser v3 ANSWER-FIRST loaded")

✅ Parser v3 ANSWER-FIRST loaded


In [5]:
def generate_reasoning(text: str, max_tokens: int = 350) -> dict:
    messages = build_prompt(text)

    formatted_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    _device = next(model.parameters()).device
    model_inputs = tokenizer(text=formatted_text, return_tensors="pt").to(_device)
    inputs       = model_inputs["input_ids"]
    attn_mask    = model_inputs["attention_mask"]

    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs,
            attention_mask=attn_mask,
            max_new_tokens=max_tokens,
            temperature=0.7,
            do_sample=True,
            use_cache=True,
            repetition_penalty=1.2,
        )

    raw = tokenizer.decode(
        outputs[0][inputs.shape[-1]:], skip_special_tokens=True
    ).strip()
    raw = raw.replace("<end_of_turn>", "").replace("<|turn>", "").strip()

    m, st, se, parsed = parse_output(raw)

    if "=== GIẢI THÍCH ===" in raw:
        reasoning = raw.split("=== GIẢI THÍCH ===")[-1].strip()
    elif "=== giải thích ===" in raw.lower():
        idx = raw.lower().find("=== giải thích ===")
        reasoning = raw[idx:].split('\n', 1)[-1].strip()
    else:
        reasoning = raw
    if not reasoning.startswith("Lý luận"):
        reasoning = "Lý luận: " + reasoning

    return {
        "reasoning":  reasoning,
        "misinfo":    LABEL_MISINFO.get(m, "?"),
        "stance":     LABEL_STANCE.get(st, "?"),
        "sentiment":  LABEL_SENTIMENT.get(se, "?"),
        "parsed":     parsed,
        "raw":        raw,
    }

# Quick test
test_result = generate_reasoning("Vắc-xin COVID gây vô sinh ở phụ nữ trẻ.")
print("=== TEST ===")
print(f"Misinfo:   {test_result['misinfo']}")
print(f"Stance:    {test_result['stance']}")
print(f"Sentiment: {test_result['sentiment']}")
print(f"Parsed:    {test_result['parsed']}")
print(f"Reasoning: {test_result['reasoning'][:200]}...")

=== TEST ===
Misinfo:   Tin giả
Stance:    Phản đối
Sentiment: Tiêu cực
Parsed:    True
Reasoning: Lý luận: **Lý luận:**

1. **Misinformation (Tin gia):** Câu này đưa ra một tuyên bố y tế nghiêm trọng ("gây vô sinh") nhưng hoàn toàn không có cơ sở khoa học hay dữ liệu lâm sàng nào xác nhận điều đó....


In [6]:
from fastapi import FastAPI
from pydantic import BaseModel
from pyngrok import ngrok, conf
import nest_asyncio
import uvicorn
import threading
import subprocess
import time
import torch

# Kill port nếu đang bị chiếm
subprocess.run(["fuser", "-k", "8000/tcp"], capture_output=True)
time.sleep(1)

conf.get_default().auth_token = NGROK_TOKEN
app = FastAPI(title="VaccineNLP Gemma-4 Inference Server")

# ⚠️ Rất quan trọng để tránh crash CUDA khi gọi qua API
model_lock = threading.Lock()

class InferenceRequest(BaseModel):
    text: str
    max_tokens: int = 350

class InferenceResponse(BaseModel):
    reasoning:  str
    misinfo:    str = "?"
    stance:     str = "?"
    sentiment:  str = "?"
    parsed:     bool = False
    status:     str = "success"
    error:      str = ""

@app.post("/predict", response_model=InferenceResponse)
def predict(req: InferenceRequest):
    try:
        if not req.text or not req.text.strip():
            return InferenceResponse(reasoning="", status="error", error="Empty text")
        
        with model_lock:
            result = generate_reasoning(req.text, req.max_tokens)
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
                
            return InferenceResponse(
                reasoning = result["reasoning"],
                misinfo   = result["misinfo"],
                stance    = result["stance"],
                sentiment = result["sentiment"],
                parsed    = result["parsed"],
            )
    except Exception as e:
        return InferenceResponse(reasoning="", status="error", error=str(e))

@app.get("/health")
def health():
    return {
        "status":     "healthy",
        "model":      "Gemma-4 E4B (VaccineNLP fine-tuned)",
        "gpu_mem_gb": round(torch.cuda.memory_allocated() / 1e9, 2),
    }

@app.get("/")
def root():
    return {"app": "VaccineNLP Gemma-4 Kaggle Host", "version": "3.0"}

nest_asyncio.apply()

print("🌐 Starting ngrok tunnel...")
public_url = ngrok.connect(8000, "http").public_url
print(f"\n{'='*60}")
print(f"🔗 PUBLIC URL:      {public_url}")
print(f"🔗 PREDICT:         {public_url}/predict")
print(f"🔗 HEALTH:          {public_url}/health")
print(f"{'='*60}\n")
print("▲ SAO CHÉP URL TRÊN — dùng cho Frontend / HF Spaces config")

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning")

threading.Thread(target=run_server, daemon=True).start()
time.sleep(3)
print("✅ Server running. Keep this cell alive!")

🌐 Starting ngrok tunnel...
                                                                                                    
🔗 PUBLIC URL:      https://pearle-staglike-nonsyntonically.ngrok-free.dev
🔗 PREDICT:         https://pearle-staglike-nonsyntonically.ngrok-free.dev/predict
🔗 HEALTH:          https://pearle-staglike-nonsyntonically.ngrok-free.dev/health

▲ SAO CHÉP URL TRÊN — dùng cho Frontend / HF Spaces config
✅ Server running. Keep this cell alive!


In [7]:
import requests

response = requests.post(
    f"{public_url}/predict",
    json={"text": "Vắc-xin COVID gây vô sinh ở phụ nữ trẻ và biến đổi gen ở trẻ em."},
    timeout=180, 
)

print(f"Status: {response.status_code}")
if response.status_code == 200:
    d = response.json()
    print(f"\n📊 Misinformation : {d['misinfo']}")
    print(f"📊 Stance         : {d['stance']}")
    print(f"📊 Sentiment      : {d['sentiment']}")
    print(f"📊 Parsed OK      : {d['parsed']}")
    print(f"\n💭 Reasoning:\n{d['reasoning']}")
else:
    print(f"❌ Error: {response.text}")

Status: 200

📊 Misinformation : Tin giả
📊 Stance         : Phản đối
📊 Sentiment      : Tiêu cực
📊 Parsed OK      : True

💭 Reasoning:
Lý luận: **Lý luận:**

1. **Misinformation (Tin gia):** Câu này đưa ra các tuyên bố cực đoan, chưa được bất kỳ cơ quan y tế uy tín nào trên thế giới (WHO, CDC, Bộ Y tế Việt Nam) xác nhận. Các nghiên cứu lâm sàng quy mô lớn đã chứng minh vắc-xin COVID-19 an toàn cho phụ nữ trong độ tuổi sinh sản và không gây đột biến gen ở trẻ em. Đây là ví dụ điển hình của thuyết âm mưu/tin giả nhằm gây hoang mang dư luận. -> Tin gia

2. **Stance (Quan điểm):** Người viết hoàn toàn phản đối việc tiêm chủng vắc-xin COVID-19 dựa trên những hậu quả tiêu cực không có thật mà họ gán cho nó. -> Phan doi

3. Sentiment (Cảm xúc):** Ngôn ngữ sử dụng ("gây vô sinh", "biến đổi gen") là cực kỳ tiêu cực, đe dọa đến tương lai của con người. -> Tieu cuc
=== END OF TURN ===


In [ ]:
import time, requests

print("🟢 Server alive — waiting for requests...")
print(f"🔗 {public_url}/predict")
print("⚠️ URL ngrok thay đổi theo session — hãy cập nhật frontend nếu bạn khởi động lại Kaggle")

while True:
    time.sleep(60)
    try:
        h = requests.get(f"{public_url}/health", timeout=10).json()
        print(f"[{time.strftime('%H:%M:%S')}] Alive | GPU: {h['gpu_mem_gb']} GB")
    except Exception as e:
        print(f"[{time.strftime('%H:%M:%S')}] ⚠️ {e}")